# PREPROCESSING - ChurnShield AI — Explainable Customer Retention & Revenue Intelligence Platform

## 1) IMPORT LIBRARIES

In [25]:

# 1. IMPORT LIBRARIES

import numpy as np
import pandas as pd
import joblib
import sys

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight


# 2. PROJECT PATH

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "churn_preprocessing.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project Root:", PROJECT_ROOT)

# Add project folder to Python search path
sys.path.insert(0, str(PROJECT_ROOT))

# Check whether Python file exists
if not (PROJECT_ROOT / "churn_preprocessing.py").exists():
    raise FileNotFoundError(
        "churn_preprocessing.py not found in: "
        + str(PROJECT_ROOT)
    )

# 3. IMPORT CUSTOM FEATURE ENGINEERING
from churn_preprocessing import ChurnFeatureEngineer

Project Root: C:\Users\ABHINDEV S\OneDrive\Desktop\ChurnShield AI\ChurnShield-AI


## 2) LOAD CLEANED DATASET

In [26]:
df = pd.read_csv("telco_cleaned.csv")

In [27]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [28]:
df.tail()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7038,Male,No,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,No,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,No,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,Yes,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes
7042,Male,No,No,No,66,Yes,No,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.50,No


In [29]:
df.shape

(7043, 20)

## DATA VALIDATION

In [30]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [31]:
df.duplicated().sum()

np.int64(22)

### 22 duplicate feature rows ഉണ്ടെന്നതുകൊണ്ട് മാത്രം അവ delete ചെയ്യുന്നില്ല. differnt customers-ന് ഒരേ feature values ഉണ്ടാകാം.

## 4) Target Encoding

In [32]:
df["ChurnFlag"] = df["Churn"].map({"No": 0,"Yes": 1})

In [33]:
if df["ChurnFlag"].isnull().any():
    raise ValueError("Unexpected values in Churn column.")

In [34]:
print("Target Distribution:")
print(df["ChurnFlag"].value_counts())

Target Distribution:
ChurnFlag
0    5174
1    1869
Name: count, dtype: int64


In [35]:
print("\nTarget Distribution (%):")
print((df["ChurnFlag"].value_counts(normalize=True) * 100).round(2))


Target Distribution (%):
ChurnFlag
0    73.46
1    26.54
Name: proportion, dtype: float64


## 5) Feature Engineering
#### Feature engineering and category simplification are handled
by the custom ChurnFeatureEngineer transformer.

## 7) REMOVE UNNECESSARY COLUMNS

In [36]:
df_model = df.drop(columns=["Churn"])
df_model.shape

(7043, 20)

#### churnFlag target നിലനിർത്തുന്നു.

## 8) SEPARATE FEATURES AND TARGET

In [37]:
X = df_model.drop(columns=["ChurnFlag"])
y = df_model["ChurnFlag"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Shape: (7043, 19)
Target Shape: (7043,)


## 9) Train-Test Split

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

y_train.name = "ChurnFlag"
y_test.name = "ChurnFlag"

In [39]:
X_train.shape

(5634, 19)

In [40]:
X_test.shape

(1409, 19)

In [41]:
print("\nTraining Target Distribution:")
print(y_train.value_counts())


Training Target Distribution:
ChurnFlag
0    4139
1    1495
Name: count, dtype: int64


## 10) Identify Column Groups

In [42]:
num_cols = ["tenure","MonthlyCharges","TotalCharges","num_addons","avg_historical_monthly_charge"]

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

passthrough_cols = ["is_new_customer"]


In [43]:
print("Numerical Columns:", num_cols)
print("\nCategorical Columns:", cat_cols)
print("\nPassthrough Columns:", passthrough_cols)

Numerical Columns: ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_addons', 'avg_historical_monthly_charge']

Categorical Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Passthrough Columns: ['is_new_customer']


# 11) Create Preprocessing Pipeline

In [44]:
numeric_transformer = StandardScaler()
try:
    categorical_transformer = OneHotEncoder(drop="if_binary",handle_unknown="ignore",sparse_output=False)

except TypeError:categorical_transformer = OneHotEncoder(drop="if_binary",handle_unknown="ignore",sparse=False)

In [45]:
categorical_transformer

,categories,'auto'
,drop,'if_binary'
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


#### Create Column Transformer

In [48]:
column_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
        ("pass", "passthrough", passthrough_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False)

In [49]:
column_preprocessor

,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


### Create Complete Pipeline

In [57]:
preprocessor = Pipeline(steps=[("feature_engineering", ChurnFeatureEngineer()),("column_preprocessing", column_preprocessor)])

preprocessor


,steps,"[('feature_engineering', ...), ('column_preprocessing', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False


## 12) Fit and Transform

In [61]:
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

feature_names = feature_names = (preprocessor.named_steps["column_preprocessing"].get_feature_names_out())

X_train_p = pd.DataFrame(X_train_p,columns=feature_names,index=X_train.index)
X_test_p = pd.DataFrame(X_test_p,columns=feature_names,index=X_test.index)

print("Training Shape:", X_train_p.shape)
print("Testing Shape:", X_test_p.shape)
print("Total Features:", len(feature_names))

Training Shape: (5634, 29)
Testing Shape: (1409, 29)
Total Features: 29


### PROCESSED DATA VALIDATION

In [62]:
print("Missing Values in Train:",
      X_train_p.isnull().sum().sum())

print("Missing Values in Test:",
      X_test_p.isnull().sum().sum())

print("\nDuplicate Rows in Train:",
      X_train_p.duplicated().sum())

print("\nFeature Count:", len(feature_names))

print("\nScaled Numerical Features:")
print(X_train_p[num_cols].describe().loc[["mean", "std"]].round(2))

Missing Values in Train: 0
Missing Values in Test: 0

Duplicate Rows in Train: 26

Feature Count: 29

Scaled Numerical Features:
      tenure  MonthlyCharges  TotalCharges  num_addons  \
mean    -0.0            -0.0           0.0         0.0   
std      1.0             1.0           1.0         1.0   

      avg_historical_monthly_charge  
mean                           -0.0  
std                             1.0  


## 14) Class Imbalance Analysis

In [63]:
print("Training Target Distribution:")
print(y_train.value_counts())

Training Target Distribution:
ChurnFlag
0    4139
1    1495
Name: count, dtype: int64


In [64]:
print("\nTraining Target Percentage:")
print((y_train.value_counts(normalize=True) * 100).round(2))


Training Target Percentage:
ChurnFlag
0    73.46
1    26.54
Name: proportion, dtype: float64


## 15) Calculate Class Weights

In [65]:
classes = np.array([0, 1])

weights = compute_class_weight(class_weight="balanced",classes=classes,y=y_train)
class_weight = {int(c): float(w)for c, w in zip(classes, weights)}
scale_pos_weight = ((y_train == 0).sum() /(y_train == 1).sum())

In [66]:
print("Class Weights:", class_weight)
print("Scale Pos Weight:", scale_pos_weight)

Class Weights: {0: 0.6805991785455424, 1: 1.8842809364548494}
Scale Pos Weight: 2.768561872909699


## 16) Save Processed Data

In [67]:
OUTPUT_DIR = PROJECT_ROOT

# Save training and testing data
X_train_p.to_csv(OUTPUT_DIR / "X_train.csv",index=False)

X_test_p.to_csv(OUTPUT_DIR / "X_test.csv",index=False)

y_train.to_csv(OUTPUT_DIR / "y_train.csv",index=False)

y_test.to_csv(OUTPUT_DIR / "y_test.csv",index=False)

# Save complete preprocessing pipeline

joblib.dump(preprocessor,OUTPUT_DIR / "preprocessor.joblib")

# Save feature names

joblib.dump(list(feature_names),OUTPUT_DIR / "feature_names.joblib")

# Save class weights

joblib.dump(class_weight,OUTPUT_DIR / "class_weight.joblib")

joblib.dump(scale_pos_weight,OUTPUT_DIR / "scale_pos_weight.joblib")


['C:\\Users\\ABHINDEV S\\OneDrive\\Desktop\\ChurnShield AI\\ChurnShield-AI\\scale_pos_weight.joblib']

## Final Validation

In [ ]:
print("X_train:", X_train_p.shape)
print("X_test :", X_test_p.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("Feature Names:", len(feature_names))

print("Train Missing Values:",X_train_p.isnull().sum().sum())

print("Test Missing Values:",X_test_p.isnull().sum().sum())

print("Target Classes:",sorted(y_train.unique()))

In [69]:
print("\nTraining Data:", X_train_p.shape)
print("Testing Data:", X_test_p.shape)

print("\nTraining Target:", y_train.shape)
print("Testing Target:", y_test.shape)

print("\nFeature Count:", len(feature_names))

print("\nTrain Missing Values:",X_train_p.isnull().sum().sum())

print("Test Missing Values:",X_test_p.isnull().sum().sum())

print("\nTarget Classes:", sorted(y_train.unique()))

print("\nSaved Files:")

saved_files = ["X_train.csv","X_test.csv","y_train.csv","y_test.csv","preprocessor.joblib","feature_names.joblib","class_weight.joblib","scale_pos_weight.joblib"]

for file_name in saved_files:
    file_path = OUTPUT_DIR / file_name
    print(file_name, ":", file_path.exists())


Training Data: (5634, 29)
Testing Data: (1409, 29)

Training Target: (5634,)
Testing Target: (1409,)

Feature Count: 29

Train Missing Values: 0
Test Missing Values: 0

Target Classes: [np.int64(0), np.int64(1)]

Saved Files:
X_train.csv : True
X_test.csv : True
y_train.csv : True
y_test.csv : True
preprocessor.joblib : True
feature_names.joblib : True
class_weight.joblib : True
scale_pos_weight.joblib : True
